# iMAPS forecast report

Locational clearance demand per barangay, from the forecasting service: the same
numbers the Map's Trends panel shows.

**Before running:** the forecasting service must be up on port 8002
(`docker compose up -d` in `../iMAPS-forecasting-service`). The URL and API key are
read from iMAPS' `.env`. Then use **Run All**.

In [ ]:
# @title Settings
TOP = 15  # @param {type:"slider", min:5, max:48, step:1}
SAVE_CHARTS = False  # @param {type:"boolean"}
SAVE_DIR = "../storage/app/forecast-report"  # @param {type:"string"}

In [ ]:
# @title Setup
%matplotlib inline
import sys
from pathlib import Path

import matplotlib.pyplot as plt

# The helpers live in forecast_report.py next to this notebook.
sys.path.insert(0, str(Path.cwd() if Path("forecast_report.py").exists() else Path.cwd() / "scripts"))
import forecast_report as fr

figures = {}

In [ ]:
# @title Run the forecast
url, api_key, timeout = fr.service_settings()
print(f"Requesting forecast from {url} ...")
result = fr.fetch_forecast(url, api_key, timeout)
fr.print_window(result)

## Validation

The model is scored on the last two quarters, which it never saw while training.
The baseline repeats each barangay's previous quarter; the model is only useful if
it beats that.

In [ ]:
# @title Model vs baseline
fr.print_metrics(result)
figures["metrics"] = fr.chart_metrics(plt, result["metrics"])
plt.show()

## Municipal totals

Quarterly clearances filed, then the next two quarters forecast with an 80% range:
the actual total should land inside the range about 4 times in 5.

In [ ]:
# @title Totals per quarter
fr.print_totals(result)
figures["totals"] = fr.chart_totals(plt, result)
plt.show()

## Barangays

Expected clearances per barangay, ranked by the first forecast quarter. Barangays
with only a few expected filings are labelled `low` confidence: at those counts a
single application swings the figure.

In [ ]:
# @title Barangay forecast
fr.print_barangays(result, TOP)
figures["barangays"] = fr.chart_barangays(plt, result, TOP)
plt.show()

In [ ]:
# @title Save charts (when SAVE_CHARTS is ticked)
if SAVE_CHARTS:
    out = Path(SAVE_DIR)
    out.mkdir(parents=True, exist_ok=True)
    for name, fig in figures.items():
        fig.savefig(out / f"forecast_{name}.png", dpi=150, facecolor=fr.SURFACE)
    print(f"Charts written to {out.resolve()}")
else:
    print("Not saved. Tick SAVE_CHARTS in Settings to write PNGs.")